# AI Career Advisor & Learning Path Generator
#Capstone Project- 10 
#Name - Mrinal Mayank (6400955)
## Multi-Agent Intelligent Advisor using LangGraph + Groq + Gradio

This notebook is a **complete Colab-ready academic mini project** that builds a multi-step intelligent career advisory system for students.

## Objective
Build an AI-based multi-agent system that:
- analyzes a student's current profile,
- recommends suitable career roles,
- identifies skill gaps,
- generates a structured learning roadmap,
- and produces final practical career advice.

## Core Agent Flow

```text
Student Input
   ↓
Profile Analyzer Agent
   ↓
Career Recommendation Agent
   ↓
Skill Gap Agent
   ↓
Conditional Branch
   ├── Beginner → Beginner Learning Path Agent
   └── Intermediate/Advanced → Accelerated Learning Path Agent
   ↓
Final Advisor Agent
   ↓
Gradio / API / Notebook Output
```

## Problem Statement
Students often know some skills but struggle to answer:
- Which career role fits me best?
- What skills am I still missing?
- What should I learn next, and in what order?

This notebook solves that problem through **multiple specialized agents** with **shared state passing**.

## 1. Required Installations
Run this cell first in Google Colab.

In [1]:
!pip -q install -U groq langgraph gradio fastapi uvicorn nest_asyncio ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.0/43.0 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.4/117.4 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.0 MB/s eta 0:00:00


## 2. Groq API Setup from Colab Secrets Only

This notebook loads the API key **only** from **Google Colab Secrets**.

### Secret name required
`GROQ_API_KEY`

### Steps
1. Open left sidebar in Colab
2. Click **Secrets**
3. Add new secret
4. Name = `GROQ_API_KEY`
5. Value = your Groq API key
6. Enable notebook access

In [2]:
# ============================================================
# GROQ API SETUP FROM COLAB SECRETS ONLY
# ============================================================

import os
from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY not found in Colab Secrets. Add it from the left sidebar > Secrets.")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

MODEL_NAME = "llama-3.3-70b-versatile"

print("Groq API key loaded from Colab Secrets.")
print("Selected model:", MODEL_NAME)

Groq API key loaded from Colab Secrets.
Selected model: llama-3.3-70b-versatile


## 3. Imports

In [3]:
# ============================================================
# IMPORTS
# ============================================================

import os
import json
import re
from typing import Any, Dict, List, TypedDict

import pandas as pd
import gradio as gr
import nest_asyncio

from groq import Groq
from langgraph.graph import StateGraph, START, END

from fastapi import FastAPI
from pydantic import BaseModel, Field
from fastapi.testclient import TestClient

from IPython.display import display, Markdown

nest_asyncio.apply()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

## 4. Scenario Explanation

This project simulates a student-facing **AI Career Advisor**.

### Input
The system takes a student's profile in natural language.

### Example
> I know Python basics and statistics, and I want to become a Data Scientist

### Output
The system returns:
1. Student Profile Summary  
2. Current Skill Assessment  
3. Recommended Career Roles  
4. Skill Gaps  
5. Personalized Learning Roadmap  
6. Final Career Advice  

### Multi-Agent Design
The workflow uses specialized agents:
- **Profile Analyzer Agent**
- **Career Recommendation Agent**
- **Skill Gap Agent**
- **Learning Path Agent**
- **Final Advisor Agent**

## 5. Student Profile Sample Dataset

In [4]:
student_profiles = [
    {
        "student_id": "S1",
        "profile_text": "I know Python basics and statistics, and I want to become a Data Scientist"
    },
    {
        "student_id": "S2",
        "profile_text": "I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer"
    }
]

pd.DataFrame(student_profiles)

,student_id,profile_text
0,S1,"I know Python basics and statistics, and I wan..."
1,S2,"I know HTML, CSS, JavaScript, and React basics..."


## 6. Career Knowledge Base

This lightweight knowledge base helps the system compare current skills against target role expectations in an explainable way.

In [5]:
ROLE_KNOWLEDGE_BASE = {
    "Data Scientist": {
        "required_skills": [
            "Python", "Statistics", "SQL", "Machine Learning",
            "Data Visualization", "Pandas", "NumPy",
            "Model Evaluation", "Feature Engineering", "Deployment"
        ],
        "tools": ["Jupyter", "scikit-learn", "Pandas", "Matplotlib", "Git"],
        "projects": [
            "Exploratory data analysis project",
            "Predictive modeling project",
            "Mini deployment project using FastAPI or Gradio"
        ]
    },
    "Data Analyst": {
        "required_skills": [
            "Excel", "SQL", "Python", "Statistics",
            "Data Cleaning", "Power BI", "Tableau", "Data Visualization"
        ],
        "tools": ["Excel", "SQL", "Power BI", "Tableau", "Python"],
        "projects": [
            "Business dashboard project",
            "Sales trend analysis project",
            "EDA and business insights case study"
        ]
    },
    "ML Engineer": {
        "required_skills": [
            "Python", "Machine Learning", "Deep Learning",
            "SQL", "APIs", "Docker", "Deployment",
            "Model Serving", "MLOps", "Git"
        ],
        "tools": ["TensorFlow", "PyTorch", "FastAPI", "Docker", "GitHub"],
        "projects": [
            "Model serving API",
            "Image classification system",
            "End-to-end ML pipeline project"
        ]
    },
    "Full Stack Developer": {
        "required_skills": [
            "HTML", "CSS", "JavaScript", "React",
            "Node.js", "Express.js", "MongoDB", "REST APIs",
            "Authentication", "Git", "Deployment"
        ],
        "tools": ["React", "Node.js", "Express.js", "MongoDB Atlas", "Postman", "Git", "Vercel"],
        "projects": [
            "MERN CRUD application",
            "Authentication-based full-stack app",
            "Deployed portfolio-grade web application"
        ]
    },
    "Frontend Developer": {
        "required_skills": [
            "HTML", "CSS", "JavaScript", "React",
            "Responsive Design", "State Management", "API Integration", "Git"
        ],
        "tools": ["React", "Tailwind CSS", "Git", "Vercel", "Figma"],
        "projects": [
            "Responsive landing page",
            "API-driven dashboard",
            "Modern UI clone project"
        ]
    },
    "Backend Developer": {
        "required_skills": [
            "JavaScript", "Node.js", "Express.js", "REST APIs",
            "Databases", "Authentication", "Git", "Deployment"
        ],
        "tools": ["Node.js", "Express.js", "Postman", "MongoDB", "Docker"],
        "projects": [
            "REST API project",
            "Authentication service",
            "Scalable backend mini project"
        ]
    }
}

## 7. Shared State Object

This object is passed between agents.  
Each agent updates some part of the state, and the next agent consumes it.

In [6]:
class CareerAdvisorState(TypedDict, total=False):
    # Input
    student_id: str
    profile_text: str

    # Extracted analysis
    extracted_skills: List[str]
    stated_goal: str
    level: str
    interests: List[str]
    profile_summary: str
    assessment: str

    # Career recommendation
    recommended_roles: List[Dict[str, Any]]
    target_role: str

    # Skill gap
    skill_gaps: List[str]
    skill_gap_summary: str

    # Roadmap
    roadmap: Dict[str, Any]

    # Final layer
    reasoning_summary: str
    final_advice: str

    # Workflow routing
    route_label: str

## 8. LLM Helper Functions

We use JSON-first prompting so each agent returns structured output.

In [7]:
def extract_json_block(text: str) -> Dict[str, Any]:
    text = text.strip()

    # Try direct JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # Try fenced JSON block
    match = re.search(r"```json\s*(\{.*?\})\s*```|```\s*(\{.*?\})\s*```", text, re.DOTALL)
    if match:
        json_text = match.group(1) or match.group(2)
        return json.loads(json_text)

    # Fallback to first JSON object
    brace_match = re.search(r"\{.*\}", text, re.DOTALL)
    if brace_match:
        return json.loads(brace_match.group(0))

    raise ValueError(f"Could not parse JSON from model output:\n{text}")


def ask_llm_for_json(system_prompt: str, user_prompt: str) -> Dict[str, Any]:
    completion = client.chat.completions.create(
        model=MODEL_NAME,
        temperature=0.2,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
    )
    raw_text = completion.choices[0].message.content.strip()
    return extract_json_block(raw_text)

## 9. Agent Definitions

These are the core agents of the system.

### Agents
1. Profile Analyzer Agent  
2. Career Recommendation Agent  
3. Skill Gap Agent  
4. Beginner Learning Path Agent  
5. Intermediate Learning Path Agent  
6. Final Advisor Agent  

In [8]:
def profile_analyzer_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    system_prompt = '''
You are the Profile Analyzer Agent in a multi-agent career advisory system.

Your job:
- Extract the student's current skills
- Identify the target career goal
- Estimate the student's level as beginner, intermediate, or advanced
- Identify interests/domains
- Create a concise profile summary
- Create a concise current skill assessment

Return ONLY valid JSON with this exact schema:
{
  "extracted_skills": ["..."],
  "stated_goal": "...",
  "level": "beginner or intermediate or advanced",
  "interests": ["..."],
  "profile_summary": "...",
  "assessment": "..."
}

Rules:
- If the student says basics, treat them as beginner unless strong evidence suggests otherwise
- Keep skills normalized and concise
'''
    result = ask_llm_for_json(system_prompt, f"Student Profile: {state['profile_text']}")

    level = result.get("level", "beginner").strip().lower()
    if level not in {"beginner", "intermediate", "advanced"}:
        level = "beginner"

    return {
        **state,
        "extracted_skills": result.get("extracted_skills", []),
        "stated_goal": result.get("stated_goal", ""),
        "level": level,
        "interests": result.get("interests", []),
        "profile_summary": result.get("profile_summary", ""),
        "assessment": result.get("assessment", ""),
        "route_label": "beginner_path" if level == "beginner" else "intermediate_path"
    }


def career_recommendation_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    role_catalog = list(ROLE_KNOWLEDGE_BASE.keys())

    system_prompt = f'''
You are the Career Recommendation Agent in a multi-agent career advisory system.

Available role catalog:
{json.dumps(role_catalog, indent=2)}

Your job:
- Recommend the top 3 most suitable roles from the role catalog
- Rank them in best-fit order
- Explain why each role fits
- Select one primary target role

Return ONLY valid JSON with this exact schema:
{{
  "recommended_roles": [
    {{
      "role": "...",
      "match_score": 0,
      "why_fit": "..."
    }}
  ],
  "target_role": "..."
}}

Rules:
- Use only role names from the provided catalog
- match_score must be an integer from 1 to 100
'''
    user_prompt = f'''
Profile summary: {state.get("profile_summary", "")}
Current skills: {state.get("extracted_skills", [])}
Goal: {state.get("stated_goal", "")}
Level: {state.get("level", "")}
Interests: {state.get("interests", [])}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)

    return {
        **state,
        "recommended_roles": result.get("recommended_roles", []),
        "target_role": result.get("target_role", "")
    }


def skill_gap_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    current_skills = {skill.strip().lower() for skill in state.get("extracted_skills", [])}
    required_skills = ROLE_KNOWLEDGE_BASE.get(target_role, {}).get("required_skills", [])

    skill_gaps = [skill for skill in required_skills if skill.strip().lower() not in current_skills]

    system_prompt = '''
You are the Skill Gap Agent.

Your job:
- Summarize the technical and practical gap between the student's current profile and the target role

Return ONLY valid JSON with this exact schema:
{
  "skill_gap_summary": "..."
}
'''
    user_prompt = f'''
Current skills: {state.get("extracted_skills", [])}
Target role: {target_role}
Required skills: {required_skills}
Computed gaps: {skill_gaps}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)

    return {
        **state,
        "skill_gaps": skill_gaps,
        "skill_gap_summary": result.get("skill_gap_summary", "")
    }


def beginner_learning_path_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    kb = ROLE_KNOWLEDGE_BASE.get(target_role, {})

    system_prompt = '''
You are the Learning Path Agent for a BEGINNER student.

Your job:
- Create a beginner-first learning roadmap
- Organize it into beginner, intermediate, and advanced stages
- Include project suggestions
- Keep the sequence realistic and easy to follow

Return ONLY valid JSON with this exact schema:
{
  "roadmap": {
    "beginner_stage": ["..."],
    "intermediate_stage": ["..."],
    "advanced_stage": ["..."],
    "suggested_projects": ["..."],
    "timeline_note": "...",
    "learning_strategy": "..."
  }
}
'''
    user_prompt = f'''
Level: {state.get("level")}
Target role: {target_role}
Current skills: {state.get("extracted_skills", [])}
Skill gaps: {state.get("skill_gaps", [])}
Relevant tools: {kb.get("tools", [])}
Project ideas: {kb.get("projects", [])}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)

    return {
        **state,
        "roadmap": result.get("roadmap", {})
    }


def intermediate_learning_path_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    target_role = state.get("target_role", "")
    kb = ROLE_KNOWLEDGE_BASE.get(target_role, {})

    system_prompt = '''
You are the Learning Path Agent for an INTERMEDIATE or ADVANCED student.

Your job:
- Create a faster roadmap with stronger applied focus
- Keep the beginner stage short
- Focus more on real projects, portfolio, deployment, and interview readiness

Return ONLY valid JSON with this exact schema:
{
  "roadmap": {
    "beginner_stage": ["..."],
    "intermediate_stage": ["..."],
    "advanced_stage": ["..."],
    "suggested_projects": ["..."],
    "timeline_note": "...",
    "learning_strategy": "..."
  }
}
'''
    user_prompt = f'''
Level: {state.get("level")}
Target role: {target_role}
Current skills: {state.get("extracted_skills", [])}
Skill gaps: {state.get("skill_gaps", [])}
Relevant tools: {kb.get("tools", [])}
Project ideas: {kb.get("projects", [])}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)

    return {
        **state,
        "roadmap": result.get("roadmap", {})
    }


def final_advisor_agent(state: CareerAdvisorState) -> CareerAdvisorState:
    system_prompt = '''
You are the Final Advisor Agent.

Your job:
- Consolidate all previous agent outputs
- Write a reasoning summary
- Give final practical career advice

Return ONLY valid JSON with this exact schema:
{
  "reasoning_summary": "...",
  "final_advice": "..."
}
'''
    user_prompt = f'''
Profile summary: {state.get("profile_summary", "")}
Assessment: {state.get("assessment", "")}
Recommended roles: {state.get("recommended_roles", [])}
Target role: {state.get("target_role", "")}
Skill gaps: {state.get("skill_gaps", [])}
Skill gap summary: {state.get("skill_gap_summary", "")}
Roadmap: {json.dumps(state.get("roadmap", {}), indent=2)}
'''
    result = ask_llm_for_json(system_prompt, user_prompt)

    return {
        **state,
        "reasoning_summary": result.get("reasoning_summary", ""),
        "final_advice": result.get("final_advice", "")
    }

## 10. Workflow / Graph Definition

This project uses **LangGraph** for sequential stateful agent orchestration.

In [9]:
def route_after_gap_analysis(state: CareerAdvisorState) -> str:
    return state.get("route_label", "beginner_path")


graph_builder = StateGraph(CareerAdvisorState)

graph_builder.add_node("profile_analyzer", profile_analyzer_agent)
graph_builder.add_node("career_recommender", career_recommendation_agent)
graph_builder.add_node("skill_gap_agent", skill_gap_agent)
graph_builder.add_node("beginner_roadmap_agent", beginner_learning_path_agent)
graph_builder.add_node("intermediate_roadmap_agent", intermediate_learning_path_agent)
graph_builder.add_node("final_advisor", final_advisor_agent)

graph_builder.add_edge(START, "profile_analyzer")
graph_builder.add_edge("profile_analyzer", "career_recommender")
graph_builder.add_edge("career_recommender", "skill_gap_agent")

graph_builder.add_conditional_edges(
    "skill_gap_agent",
    route_after_gap_analysis,
    {
        "beginner_path": "beginner_roadmap_agent",
        "intermediate_path": "intermediate_roadmap_agent"
    }
)

graph_builder.add_edge("beginner_roadmap_agent", "final_advisor")
graph_builder.add_edge("intermediate_roadmap_agent", "final_advisor")
graph_builder.add_edge("final_advisor", END)

career_advisor_graph = graph_builder.compile()

print("LangGraph workflow compiled successfully.")

LangGraph workflow compiled successfully.


## 11. Agent Flow Diagram

In [10]:
mermaid_diagram = '''
flowchart TD
    A[Student Input] --> B[Profile Analyzer Agent]
    B --> C[Career Recommendation Agent]
    C --> D[Skill Gap Agent]
    D --> E{Student Level?}
    E -->|Beginner| F[Beginner Learning Path Agent]
    E -->|Intermediate / Advanced| G[Accelerated Learning Path Agent]
    F --> H[Final Advisor Agent]
    G --> H[Final Advisor Agent]
    H --> I[Final Career Advice]
'''
print(mermaid_diagram)


flowchart TD
    A[Student Input] --> B[Profile Analyzer Agent]
    B --> C[Career Recommendation Agent]
    C --> D[Skill Gap Agent]
    D --> E{Student Level?}
    E -->|Beginner| F[Beginner Learning Path Agent]
    E -->|Intermediate / Advanced| G[Accelerated Learning Path Agent]
    F --> H[Final Advisor Agent]
    G --> H[Final Advisor Agent]
    H --> I[Final Career Advice]



## 12. Core Runner Function

This is the main function that runs the full multi-agent workflow for one student input.

In [11]:
def run_career_advisor(student_profile: str, student_id: str = "CUSTOM") -> Dict[str, Any]:
    initial_state: CareerAdvisorState = {
        "student_id": student_id,
        "profile_text": student_profile
    }
    result = career_advisor_graph.invoke(initial_state)
    return result

## 13. Professional Output Formatting

In [12]:
def bullet_list(items: List[str]) -> str:
    if not items:
        return "- Not available"
    return "\n".join([f"- {item}" for item in items])

def render_roles(roles: List[Dict[str, Any]]) -> str:
    if not roles:
        return "- No recommendations generated"
    lines = []
    for i, role in enumerate(roles, start=1):
        lines.append(
            f"{i}. **{role.get('role', 'Unknown')}** (Match Score: {role.get('match_score', 'N/A')})  \n"
            f"   - Why fit: {role.get('why_fit', '')}"
        )
    return "\n".join(lines)

def render_roadmap(roadmap: Dict[str, Any]) -> str:
    if not roadmap:
        return "Roadmap not generated."

    return f'''
### Beginner Stage
{bullet_list(roadmap.get("beginner_stage", []))}

### Intermediate Stage
{bullet_list(roadmap.get("intermediate_stage", []))}

### Advanced Stage
{bullet_list(roadmap.get("advanced_stage", []))}

### Suggested Projects
{bullet_list(roadmap.get("suggested_projects", []))}

### Timeline Note
{roadmap.get("timeline_note", "Not provided")}

### Learning Strategy
{roadmap.get("learning_strategy", "Not provided")}
'''

def format_result_markdown(result: Dict[str, Any]) -> str:
    return f'''
# Final Advisory Result — {result.get("student_id", "Unknown Student")}

## 1. Student Profile Summary
**Original Input:** {result.get("profile_text", "")}

**Profile Summary:** {result.get("profile_summary", "")}

## 2. Current Skill Assessment
- **Extracted Skills:** {", ".join(result.get("extracted_skills", [])) or "Not available"}
- **Target Goal:** {result.get("stated_goal", "")}
- **Estimated Level:** {result.get("level", "").title()}
- **Assessment:** {result.get("assessment", "")}

## 3. Recommended Career Roles
{render_roles(result.get("recommended_roles", []))}

## 4. Skill Gaps
{bullet_list(result.get("skill_gaps", []))}

**Skill Gap Summary:** {result.get("skill_gap_summary", "")}

## 5. Personalized Learning Roadmap
{render_roadmap(result.get("roadmap", {}))}

## 6. Final Career Advice
{result.get("final_advice", "")}

## Reasoning Summary
{result.get("reasoning_summary", "")}
'''

## 14. Execution on At Least 2 Student Profiles

In [13]:
all_results = []

for profile in student_profiles:
    result = run_career_advisor(profile["profile_text"], profile["student_id"])
    all_results.append(result)

for result in all_results:
    display(Markdown(format_result_markdown(result)))
    display(Markdown("---"))


# Final Advisory Result — S1

## 1. Student Profile Summary
**Original Input:** I know Python basics and statistics, and I want to become a Data Scientist

**Profile Summary:** A beginner with Python and statistics knowledge, aiming to become a Data Scientist

## 2. Current Skill Assessment
- **Extracted Skills:** Python, Statistics
- **Target Goal:** Data Scientist
- **Estimated Level:** Beginner
- **Assessment:** Basic understanding of Python and statistics, requires further development in data science and machine learning

## 3. Recommended Career Roles
1. **Data Analyst** (Match Score: 80)  
   - Why fit: As a beginner with Python and statistics knowledge, the Data Analyst role is a suitable starting point to gain hands-on experience in data analysis and build a strong foundation for future growth in Data Science.
2. **Data Scientist** (Match Score: 90)  
   - Why fit: Given the goal of becoming a Data Scientist, this role is a direct match, and the existing skills in Python and statistics provide a solid foundation to start learning and growing in this field.
3. **ML Engineer** (Match Score: 70)  
   - Why fit: With interests in Data Science and Machine Learning, the ML Engineer role could be a good fit, but it may require additional skills in software engineering and machine learning frameworks, making it a slightly less direct match compared to Data Analyst or Data Scientist.

## 4. Skill Gaps
- SQL
- Machine Learning
- Data Visualization
- Pandas
- NumPy
- Model Evaluation
- Feature Engineering
- Deployment

**Skill Gap Summary:** The student lacks skills in SQL, Machine Learning, Data Visualization, Pandas, NumPy, Model Evaluation, Feature Engineering, and Deployment to become a Data Scientist, despite having a foundation in Python and Statistics.

## 5. Personalized Learning Roadmap

### Beginner Stage
- Learn SQL basics
- Introduction to Pandas and NumPy
- Data Visualization with Matplotlib
- Introduction to Machine Learning with scikit-learn
- Basic data preprocessing and feature engineering

### Intermediate Stage
- Advanced Machine Learning techniques
- Model Evaluation and Hyperparameter tuning
- In-depth Feature Engineering
- Data Visualization best practices
- Introduction to Deployment with FastAPI or Gradio

### Advanced Stage
- Advanced Deployment techniques
- Model serving and monitoring
- Working with large datasets
- Advanced data preprocessing and feature engineering
- Specialized Machine Learning topics (e.g. Deep Learning, Natural Language Processing)

### Suggested Projects
- Exploratory data analysis project
- Predictive modeling project
- Mini deployment project using FastAPI or Gradio

### Timeline Note
Beginner stage: 2-3 months, Intermediate stage: 3-4 months, Advanced stage: 4-6 months

### Learning Strategy
Focus on building a strong foundation in beginner stage, then gradually move to more advanced topics, practicing with projects and real-world datasets


## 6. Final Career Advice
To become a successful Data Scientist, focus on building a strong foundation in the beginner stage by learning SQL, Pandas, NumPy, Data Visualization, and introductory Machine Learning concepts. Gradually move to more advanced topics, such as Model Evaluation, Feature Engineering, and Deployment, while practicing with projects and real-world datasets. Allocate 2-3 months for the beginner stage, 3-4 months for the intermediate stage, and 4-6 months for the advanced stage. Prioritize hands-on experience, and consider taking on projects, such as exploratory data analysis, predictive modeling, and mini deployment projects, to reinforce learning and demonstrate skills to potential employers.

## Reasoning Summary
The individual has a basic understanding of Python and statistics, but requires further development in data science and machine learning to become a Data Scientist. The recommended roles, including Data Analyst, Data Scientist, and ML Engineer, highlight the need for additional skills in areas such as SQL, Machine Learning, Data Visualization, and Deployment. The provided roadmap outlines a structured approach to addressing these skill gaps, with a focus on building a strong foundation in the beginner stage, followed by more advanced topics and practical projects.


---


# Final Advisory Result — S2

## 1. Student Profile Summary
**Original Input:** I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer

**Profile Summary:** A beginner with basic knowledge of front-end development technologies, aiming to become a full stack developer.

## 2. Current Skill Assessment
- **Extracted Skills:** HTML, CSS, JavaScript, React
- **Target Goal:** Full Stack Developer
- **Estimated Level:** Beginner
- **Assessment:** Has a solid foundation in front-end development basics, but needs to expand skill set to achieve full stack development goals.

## 3. Recommended Career Roles
1. **Full Stack Developer** (Match Score: 90)  
   - Why fit: The user's goal is to become a Full Stack Developer, and they already have basic knowledge of front-end development technologies, making this role a natural fit.
2. **Frontend Developer** (Match Score: 80)  
   - Why fit: The user has current skills in HTML, CSS, JavaScript, and React, which are all relevant to front-end development, and their interests include front-end development.
3. **Backend Developer** (Match Score: 60)  
   - Why fit: Although the user's current skills and interests are focused on front-end development, becoming a Full Stack Developer will require knowledge of back-end development, making this role a potential area for growth and exploration.

## 4. Skill Gaps
- Node.js
- Express.js
- MongoDB
- REST APIs
- Authentication
- Git
- Deployment

**Skill Gap Summary:** The student lacks skills in backend development with Node.js and Express.js, database management with MongoDB, API design with REST APIs, security with Authentication, version control with Git, and software deployment, which are essential for a Full Stack Developer role.

## 5. Personalized Learning Roadmap

### Beginner Stage
- Learn Node.js basics
- Understand Express.js framework
- Introduction to MongoDB and NoSQL databases
- Learn REST APIs fundamentals
- Basic Git version control
- Deploy a simple React app to Vercel

### Intermediate Stage
- Build a simple CRUD application using MERN stack
- Implement authentication in a full-stack app
- Learn advanced MongoDB concepts and data modeling
- Understand API security and best practices
- Use Postman for API testing and debugging
- Deploy a full-stack app to a cloud platform

### Advanced Stage
- Optimize and scale a full-stack application
- Implement advanced authentication and authorization
- Learn about microservices architecture and containerization
- Use MongoDB Atlas for cloud-based database management
- Master Git workflows and collaboration
- Deploy a portfolio-grade web application with CI/CD pipeline

### Suggested Projects
- MERN CRUD application
- Authentication-based full-stack app
- Deployed portfolio-grade web application

### Timeline Note
Assuming 1-2 hours of learning per day, the beginner stage should take around 2-3 months, intermediate stage around 3-4 months, and advanced stage around 4-6 months

### Learning Strategy
Focus on building projects and applying concepts learned, while also dedicating time to learning theory and best practices


## 6. Final Career Advice
To become a successful Full Stack Developer, focus on building projects and applying concepts learned, while also dedicating time to learning theory and best practices. Start by learning Node.js, Express.js, MongoDB, and REST APIs, and then move on to building a simple CRUD application and implementing authentication. As you progress, focus on advanced topics like API security, Git workflows, and deployment. Practice by working on suggested projects, such as a MERN CRUD application and a deployed portfolio-grade web application. With consistent effort and a dedication to learning, you can fill your skill gaps and achieve your goal of becoming a Full Stack Developer within 9-13 months.

## Reasoning Summary
The user has a solid foundation in front-end development basics but needs to expand their skill set to achieve full stack development goals. The recommended roles suggest that the user is a good fit for a Full Stack Developer role, with a match score of 90. However, the user lacks skills in backend development, database management, API design, security, version control, and software deployment. The provided roadmap outlines a clear learning path, starting with beginner-stage topics such as Node.js, Express.js, MongoDB, and REST APIs, followed by intermediate-stage topics like building a CRUD application and implementing authentication, and finally advanced-stage topics such as optimizing and scaling a full-stack application.


---

## 15. State Passing Demonstration

This block shows what key values are being passed through the workflow state.

In [14]:
for result in all_results:
    compact_state = {
        "student_id": result.get("student_id"),
        "extracted_skills": result.get("extracted_skills"),
        "stated_goal": result.get("stated_goal"),
        "level": result.get("level"),
        "target_role": result.get("target_role"),
        "skill_gaps": result.get("skill_gaps"),
        "route_label": result.get("route_label"),
        "roadmap_keys": list(result.get("roadmap", {}).keys()) if result.get("roadmap") else []
    }
    print(json.dumps(compact_state, indent=2))
    print("=" * 100)

{
  "student_id": "S1",
  "extracted_skills": [
    "Python",
    "Statistics"
  ],
  "stated_goal": "Data Scientist",
  "level": "beginner",
  "target_role": "Data Scientist",
  "skill_gaps": [
    "SQL",
    "Machine Learning",
    "Data Visualization",
    "Pandas",
    "NumPy",
    "Model Evaluation",
    "Feature Engineering",
    "Deployment"
  ],
  "route_label": "beginner_path",
  "roadmap_keys": [
    "beginner_stage",
    "intermediate_stage",
    "advanced_stage",
    "suggested_projects",
    "timeline_note",
    "learning_strategy"
  ]
}
{
  "student_id": "S2",
  "extracted_skills": [
    "HTML",
    "CSS",
    "JavaScript",
    "React"
  ],
  "stated_goal": "Full Stack Developer",
  "level": "beginner",
  "target_role": "Full Stack Developer",
  "skill_gaps": [
    "Node.js",
    "Express.js",
    "MongoDB",
    "REST APIs",
    "Authentication",
    "Git",
    "Deployment"
  ],
  "route_label": "beginner_path",
  "roadmap_keys": [
    "beginner_stage",
    "intermediate_

## 16. Custom Input Cell

You can change the input below and test your own student profile.

In [15]:
custom_student_profile = "I know Python basics and statistics, and I want to become a Data Scientist"

custom_result = run_career_advisor(custom_student_profile, "CUSTOM")
display(Markdown(format_result_markdown(custom_result)))


# Final Advisory Result — CUSTOM

## 1. Student Profile Summary
**Original Input:** I know Python basics and statistics, and I want to become a Data Scientist

**Profile Summary:** A beginner with Python and statistics knowledge, aiming to become a Data Scientist.

## 2. Current Skill Assessment
- **Extracted Skills:** Python, Statistics
- **Target Goal:** Data Scientist
- **Estimated Level:** Beginner
- **Assessment:** Basic understanding of Python and statistics, requires further development in data science and machine learning skills.

## 3. Recommended Career Roles
1. **Data Analyst** (Match Score: 80)  
   - Why fit: As a beginner with Python and statistics knowledge, the Data Analyst role is a suitable starting point to gain experience in data handling and analysis, which can later be leveraged to transition into a Data Scientist role.
2. **Data Scientist** (Match Score: 90)  
   - Why fit: Given the user's goal of becoming a Data Scientist, this role is a direct match. However, considering the beginner level, it might require additional skills and experience, but it aligns perfectly with the user's interests and aspirations.
3. **ML Engineer** (Match Score: 70)  
   - Why fit: With interests in Data Science and Machine Learning, the ML Engineer role could be a good fit, although it might require additional skills in software engineering. It's a related field where the user can apply their knowledge of statistics and Python.

## 4. Skill Gaps
- SQL
- Machine Learning
- Data Visualization
- Pandas
- NumPy
- Model Evaluation
- Feature Engineering
- Deployment

**Skill Gap Summary:** The student lacks 8 essential skills to become a Data Scientist, including SQL, Machine Learning, Data Visualization, Pandas, NumPy, Model Evaluation, Feature Engineering, and Deployment, indicating a significant technical and practical gap in their current profile.

## 5. Personalized Learning Roadmap

### Beginner Stage
- Learn SQL basics
- Introduction to Pandas and NumPy
- Data Visualization with Matplotlib
- Introduction to Machine Learning with scikit-learn
- Basic data preprocessing and feature engineering

### Intermediate Stage
- Advanced Machine Learning techniques
- Model Evaluation and hyperparameter tuning
- In-depth Feature Engineering
- Data Visualization with advanced tools
- Introduction to Deployment with FastAPI or Gradio

### Advanced Stage
- Specialized Machine Learning topics (e.g., Deep Learning, Natural Language Processing)
- Advanced Deployment techniques (e.g., containerization, cloud deployment)
- Big Data processing and NoSQL databases
- Advanced data preprocessing and feature engineering techniques
- Specialized Data Visualization tools (e.g., Tableau, Power BI)

### Suggested Projects
- Exploratory data analysis project
- Predictive modeling project
- Mini deployment project using FastAPI or Gradio

### Timeline Note
Beginner stage: 2-3 months, Intermediate stage: 3-4 months, Advanced stage: 4-6 months

### Learning Strategy
Focus on building a strong foundation in beginner stage, then gradually move to more advanced topics, and practice with projects


## 6. Final Career Advice
To become a Data Scientist, focus on building a strong foundation in the beginner stage by learning SQL, Pandas, NumPy, and introductory Machine Learning concepts. Progress to the intermediate stage by developing skills in Advanced Machine Learning, Model Evaluation, and Data Visualization. Practice with projects, such as exploratory data analysis and predictive modeling, and work on a mini deployment project using FastAPI or Gradio. Allocate 2-3 months for the beginner stage, 3-4 months for the intermediate stage, and 4-6 months for the advanced stage. Stay committed to the learning strategy, and with dedication and persistence, you can bridge the skill gaps and achieve your goal of becoming a Data Scientist.

## Reasoning Summary
The user has a basic understanding of Python and statistics but requires further development in data science and machine learning skills to become a Data Scientist. The recommended roles, including Data Analyst, Data Scientist, and ML Engineer, highlight the need for the user to gain experience in data handling and analysis, and to develop additional skills in software engineering. The skill gaps identified, such as SQL, Machine Learning, and Data Visualization, indicate a significant technical and practical gap in the user's current profile. The provided roadmap outlines a structured approach to addressing these gaps, starting with beginner-level skills like SQL, Pandas, and NumPy, and progressing to more advanced topics like Model Evaluation, Feature Engineering, and Deployment.


## 17. FastAPI Layer

This section adds a lightweight API wrapper for the multi-agent workflow.

In [16]:
app = FastAPI(title="AI Career Advisor API")

class CareerRequest(BaseModel):
    student_profile: str = Field(..., description="Student profile in natural language")
    student_id: str = Field(default="API_USER", description="Optional student ID")

@app.get("/")
def root():
    return {"message": "AI Career Advisor API is running"}

@app.post("/career-advice")
def get_career_advice(payload: CareerRequest):
    result = run_career_advisor(
        student_profile=payload.student_profile,
        student_id=payload.student_id
    )
    return result

# Quick local test in notebook
client_api = TestClient(app)

api_test_payload = {
    "student_profile": "I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer",
    "student_id": "API_DEMO"
}

api_response = client_api.post("/career-advice", json=api_test_payload)
print(api_response.status_code)
print(json.dumps(api_response.json(), indent=2))

200
{
  "student_id": "API_DEMO",
  "profile_text": "I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer",
  "extracted_skills": [
    "HTML",
    "CSS",
    "JavaScript",
    "React"
  ],
  "stated_goal": "Full Stack Developer",
  "level": "beginner",
  "interests": [
    "Web Development",
    "Front-end Development"
  ],
  "profile_summary": "A beginner with basic knowledge of front-end development technologies, aiming to become a Full Stack Developer.",
  "assessment": "Has a solid foundation in front-end development basics, but needs to expand skill set to achieve full stack capabilities.",
  "recommended_roles": [
    {
      "role": "Full Stack Developer",
      "match_score": 90,
      "why_fit": "The user has expressed a clear goal of becoming a Full Stack Developer and already possesses basic front-end development skills, making this role a natural fit."
    },
    {
      "role": "Frontend Developer",
      "match_score": 80,
      "wh

## 18. Gradio UI

This section creates a simple UI so the system can be tested interactively.

In [ ]:
def gradio_career_advisor(student_profile: str):
    result = run_career_advisor(student_profile, "GRADIO_USER")
    return format_result_markdown(result)

career_demo_examples = [
    ["I know Python basics and statistics, and I want to become a Data Scientist"],
    ["I know HTML, CSS, JavaScript, and React basics, and I want to become a Full Stack Developer"],
    ["I know Python, SQL, and data visualization, and I want to become a Data Analyst"],
]

career_demo = gr.Interface(
    fn=gradio_career_advisor,
    inputs=gr.Textbox(
        lines=5,
        label="Student Profile",
        placeholder="Example: I know Python basics and statistics, and I want to become a Data Scientist"
    ),
    outputs=gr.Markdown(label="Career Advice Output"),
    title="AI Career Advisor & Learning Path Generator",
    description="Enter a student profile. The multi-agent system will analyze the profile, recommend roles, identify skill gaps, and generate a learning roadmap.",
    examples=career_demo_examples
)


career_demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c0481ec065dc2d0580.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 19. How to Give Input

This notebook accepts **one main user input**:

### Input
`student_profile`

### Example input
```text
I know Python basics and statistics, and I want to become a Data Scientist
```

### Where you can give input
- **Custom input test cell**
- **FastAPI payload**
- **Gradio UI textbox**

### Agents involved
- Profile Analyzer Agent
- Career Recommendation Agent
- Skill Gap Agent
- Learning Path Agent
- Final Advisor Agent

## 20. Conclusion and Real-World Use Cases

### How the multi-agent workflow works
This project uses a **sequential multi-agent architecture** in which each agent solves one focused task:
- the first agent understands the student,
- the second suggests relevant roles,
- the third finds missing skills,
- the fourth builds the roadmap,
- and the final agent consolidates the recommendation.

The workflow becomes strong because of:
- **shared state passing**
- **agent specialization**
- **conditional routing**
- **structured output**

### Why this is useful for students
Students usually have partial skills but not a clear roadmap.  
This system converts vague career confusion into a concrete, step-by-step plan.

### Real-world use cases
This can evolve into:
- an EdTech guidance assistant
- a college placement support tool
- a career path recommender
- a portfolio planning system
- a resume and interview readiness platform

### Extensibility
Future agents can be added easily:
- Resume Agent
- Interview Prep Agent
- Project Recommender Agent
- Certification Advisor Agent
- Job Matching Agent

This is why the design is practical, scalable, and product-ready.